## Final Project: Acquiring and processing information on world's largest banks

Importing required libraries

In [44]:
import pandas as pd
import requests
import sqlite3 as sql
from bs4 import BeautifulSoup
from datetime import datetime as dt

Adding Required Variables and Attributes

In [45]:
url = "https://web.archive.org/web/20230908091635/https://en.wikipedia.org/wiki/List_of_largest_banks"
table_attributes = ["Name", "MC_USD_Billion"]
table_attributes_final = ["Name", "MC_USD_Billion", "MC_GBP_Billion", "MC_EUR_Billion", "MC_INR_Billion"]
outputCSVPath = "./Largest_banks_data.csv"
db_name ="Banks.db"
table_name_bank_largest = "Largest_banks"
inputCSVPath = "./exchange_rate.csv"

1.1 Logging the ETL progress in a file and printing it as well.

In [46]:
def log_progress(message):
    time_format = "%Y-%m-%d %H:%M:%S"
    now = dt.now().strftime(time_format)
    with open("./code_log.txt", "a") as log:
        log.write(now + ": " + message + "\n")
    print(now,message) # print logs into terminal reference purpose

Webscraping function -> to extract data from url and converting into dataframe

In [47]:
def extract (url):
    html_page = requests.get(url).text
    html_parser = BeautifulSoup(html_page, 'html.parser')
    tables = html_parser.find_all('tbody')
    rows = tables[0].find_all('tr')
    data_list = []
    for row in rows:
        col = row.find_all('td')
        if len(col)!=0 :
            data_dict = {
                "Bank name": col[1].text.strip(),
                "Market cap US$ Billion": col[2].text.strip()
            }
            data_list.append(data_dict)
    df = pd.DataFrame(data_list)
    print("Data Frame",df)
    return df
extract(url)

Data Frame                                  Bank name Market cap US$ Billion
0                           JPMorgan Chase                 432.92
1                          Bank of America                 231.52
2  Industrial and Commercial Bank of China                 194.56
3               Agricultural Bank of China                 160.68
4                                HDFC Bank                 157.91
5                              Wells Fargo                 155.87
6                        HSBC Holdings PLC                 148.90
7                           Morgan Stanley                 140.83
8                  China Construction Bank                 139.82
9                            Bank of China                 136.81


,Bank name,Market cap US$ Billion
0,JPMorgan Chase,432.92
1,Bank of America,231.52
2,Industrial and Commercial Bank of China,194.56
3,Agricultural Bank of China,160.68
4,HDFC Bank,157.91
5,Wells Fargo,155.87
6,HSBC Holdings PLC,148.90
7,Morgan Stanley,140.83
8,China Construction Bank,139.82
9,Bank of China,136.81


Function to Load Data from exchange.csv file

In [48]:
def from_csv(input_file):
    df = pd.read_csv(input_file)
    df = df.set_index('Currency').to_dict()['Rate']
    return df

Function to transform data according to requirements

In [57]:
def transform (df, table_attributes_final, df_exchange):
    # Name
    df[table_attributes_final[0]] = df['Bank name']
    # Market cap US$ Billion
    df['Market cap US$ Billion'] = pd.to_numeric(df['Market cap US$ Billion'])
    df[table_attributes_final[1]] = df['Market cap US$ Billion']
    # Market Capitalization in GBP
    df[table_attributes_final[2]] = (df['Market cap US$ Billion'] * df_exchange['GBP']).round(2)
    # 'Market Capitalization in EUR'
    df[table_attributes_final[3]] = (df['Market cap US$ Billion'] * df_exchange['EUR']).round(2)
    #'Market Capitalization in INR'
    df[table_attributes_final[4]] = ((df['Market cap US$ Billion']) * df_exchange['INR']).round(2)
    print(df)
    return df


Function to Load transformed Datafrane into csv file

In [50]:
def load_to_csv(df,output_path):
    df.to_csv(output_path, index = False)

Function to Load transformed Datafrane into local database

In [51]:
def load_to_db(df,sql_connection, table_name):
    df.to_sql(table_name, sql_connection, index = False)

Function to run database query statements

In [52]:
def run_query(query,sql_connection):
    print(query)
    query_data = pd.read_sql_query(query,sql_connection)
    print(query_data)

Performing ETL process on the given website

In [58]:
log_progress("ETL Process Started")
log_progress("Extraction data from website Started")
extracted_df = extract(url)
log_progress("Extraction data from website ended")
log_progress("Extraction data from csv for exchanging data")
df_exchange = from_csv(inputCSVPath)
log_progress("Transformation of data Started")
transform_df = transform(extracted_df, table_attributes_final, df_exchange)
log_progress("Transformation of data ended")
log_progress("Loading data into csv Started.")
load_to_csv(transform_df, outputCSVPath)
log_progress("Loading data into csv successfully.")
log_progress("Loading data into local database started.")
sql_connection = sql.connect(db_name)
load_to_db(transform_df, sql_connection, table_name_bank_largest)
log_progress("Loading data into local database successfully.")
# Perform queries on the loaded database
statement1 = f"SELECT * FROM {table_name_bank_largest}"
run_query(statement1, sql_connection)
statement2 = f"SELECT AVG(MC_GBP_Billion) FROM {table_name_bank_largest}"
run_query(statement2, sql_connection)
statement3 = f"SELECT Name from {table_name_bank_largest} LIMIT 5"
run_query(statement3, sql_connection)
#close sql connection
sql_connection.close()
log_progress("ETL Process Completed.")

2026-04-09 00:11:50 ETL Process Started
2026-04-09 00:11:50 Extraction data from website Started
Data Frame                                  Bank name Market cap US$ Billion
0                           JPMorgan Chase                 432.92
1                          Bank of America                 231.52
2  Industrial and Commercial Bank of China                 194.56
3               Agricultural Bank of China                 160.68
4                                HDFC Bank                 157.91
5                              Wells Fargo                 155.87
6                        HSBC Holdings PLC                 148.90
7                           Morgan Stanley                 140.83
8                  China Construction Bank                 139.82
9                            Bank of China                 136.81
2026-04-09 00:11:51 Extraction data from website ended
2026-04-09 00:11:51 Extraction data from csv for exchanging data
{'EUR': 0.93, 'GBP': 0.8, 'INR': 82.95}
2026-0